In [2]:
import pandas as pd

df=pd.read_csv("../data/digikala-comments.csv")
print(df.shape)
df.head()

(6156289, 15)


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate
0,53672599,پیشنهاد نمیشود,به درد نمیخوره,23 شهریور 1402,1.0,not_recommended,True,252058,NaN,NaN,0,0,دیجی‌کالا,5A52N,NaN
1,9897229,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بن...,16 تیر 1399,0.0,recommended,True,252058,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت'],1,0,دیجی‌کالا,5A52N,NaN
2,38074516,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خ...,26 مرداد 1401,0.0,recommended,True,3331597,NaN,NaN,0,0,آرالیا بیوتی,ADM47,NaN
3,18628562,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد ...,28 اسفند 1399,0.0,recommended,True,3331329,NaN,NaN,0,0,اینجاست آ,9ZMCZ,NaN
4,53301258,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین او...,12 شهریور 1402,3.0,recommended,True,3255700,NaN,NaN,0,0,گالری آرایشی به سیما,CDWHA,NaN


In [3]:
df["recommendation_status"]=df["recommendation_status"].map({"no_idea":2, "not_recommended":0, "recommended":1})
print(df.recommendation_status.value_counts())
df1=df[df.recommendation_status==0].sample(n=5000, axis=0)
df2=df[df.recommendation_status==1].sample(n=5000, axis=0)
df3=df[df.recommendation_status==2].sample(n=5000, axis=0)

df=pd.concat([df1, df2], axis=0)
df.dropna(axis=0, inplace=True, subset=["recommendation_status", "body"])
df.recommendation_status=df.recommendation_status.astype(int)
df=df[["body", "recommendation_status"]].copy()
df.head()

recommendation_status
1.0    4162839
2.0     594121
0.0     504903
Name: count, dtype: int64


,body,recommendation_status
4105681,نخرید اصلا کیفیت نداره,0
2503402,بی کیفیت نسبت به قیمت,0
4191843,اصلل به درد من نخورد متاسفانه باز کرده بودم و ...,0
2111425,چیزی که با عکس مشاهده میکنید فرق داره,0
3107891,پیچش خراب بود و خیلی کوچیکه,0


In [4]:
import hazm

normalizer=hazm.Normalizer()
word_tokenizer=hazm.WordTokenizer()
lemmatizer=hazm.Lemmatizer()

def clean_stopword(stopword):
	return stopword.replace("آ", "ا")

stopwords=hazm.utils.stopwords_list()
stopwords=list(map(clean_stopword, stopwords))


for word in ["عالی", "خوب", "خوبی", "بی"]: stopwords.remove(word)

In [5]:
import string
import emoji
import numpy as np

def remove_chars(text, chars):
	for char in chars: text=text.replace(char, " ")
	return text

def preprocess(text):
	if not isinstance(text, str): return np.nan

	text=text.lower()
	text=remove_chars(text, string.punctuation+"؟،")

	text=emoji.replace_emoji(text, replace=" ")

	normalized_txt=normalizer.normalize(text).replace("آ", "ا")
	normalized_txt=remove_chars(normalized_txt, "۰۱۲۳۴۵۶۷۸۹\u200c")
	
	tokens=word_tokenizer.tokenize(normalized_txt)

	tokens=list(map(
		lambda token: lemmatizer.lemmatize(token).split("#")[0],
		tokens
	))

	tokens=list(filter(
		lambda token: token not in stopwords,
		tokens
	))

	if len(tokens)==0: return np.nan

	return " ".join(tokens)


df.body=df.body.apply(preprocess)
df.head()

,body,recommendation_status
4105681,خرید اصلا کیفیت نداره,0
2503402,بی کیفیت قیمت,0
4191843,اصلل درد خورد نتونستم مرجوع,0
2111425,عکس مشاهده فرق داره,0
3107891,پیچش خراب کوچیکه,0


In [6]:
print(df.isnull().sum())
df.dropna(inplace=True)
print(df.isnull().sum())
df.recommendation_status.value_counts(normalize=True)

body                     15
recommendation_status     0
dtype: int64
body                     0
recommendation_status    0
dtype: int64


recommendation_status
0    0.5003
1    0.4997
Name: proportion, dtype: float64

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
	df.body, df.recommendation_status,
	test_size=0.2,
	random_state=18
)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
	("vectorizer", TfidfVectorizer(ngram_range=(1, 2))),
	("model", LogisticRegression())
])

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
	"vectorizer__max_df": [0.75, 0.90, 1.0],
	"vectorizer__min_df": [1, 2, 5],
	"vectorizer__sublinear_tf": [True, False],
    "model__C": [0.1, 1, 10],
    "model__solver": ["lbfgs"],
    "model__penalty": ["l2"],
}

grid = GridSearchCV(
	pipeline,
	param_grid,
	cv=5,
	scoring="accuracy",
	n_jobs=-1
)

grid.fit(X_train, y_train)
print(grid.best_score_)
print(grid.best_params_)

0.8760495077205931
{'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'lbfgs', 'vectorizer__max_df': 0.75, 'vectorizer__min_df': 2, 'vectorizer__sublinear_tf': False}


f:\Roham\projects\Web\Django\comment intelligence api\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [10]:
pipeline = Pipeline([
	("vectorizer", TfidfVectorizer(
		ngram_range=(1, 2),
        max_df=0.75,
        min_df=5,
        sublinear_tf=False
	)),
	("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs"
	))
])

pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)

f:\Roham\projects\Web\Django\comment intelligence api\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


0.8582874311467201

In [11]:
pipeline.fit(df.body, df.recommendation_status)

import joblib

joblib.dump(
    pipeline,
    "../web/services/models/recommendation_detection_pipeline.joblib"
)

f:\Roham\projects\Web\Django\comment intelligence api\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


['../web/services/models/recommendation_detection_pipeline.joblib']